In [ ]:
import cv2
from ultralytics import YOLO
import math
import numpy as np
from datetime import datetime

# Para Jupyter/Colab (opcional)
try:
    import IPython.display as ipd
    from IPython.display import display, Image as IPImage
    JUPYTER = True
except:
    JUPYTER = False

In [2]:
!pip install opencv-python

In [3]:
import cv2
from ultralytics import YOLO
import math
import numpy as np

# ── Función para calcular ángulo ──────────────────────────────────────────────
def calcular_angulo(A, B, C):
    """
    Calcula el ángulo en el punto B formado por los puntos A-B-C
    """
    radianes = math.atan2(C[1] - B[1], C[0] - B[0]) - \
               math.atan2(A[1] - B[1], A[0] - B[0])
    angulo = abs(radianes * 180.0 / math.pi)
    if angulo > 180.0:
        angulo = 360 - angulo
    return angulo


# ── Función para verificar alineación corporal ────────────────────────────────
def verificar_alineacion(hombro, cadera, tobillo):
    """
    Verifica que el cuerpo esté alineado (espalda recta)
    Retorna True si la alineación es correcta
    """
    angulo_espalda = calcular_angulo(tobillo, cadera, hombro)
    return 160 <= angulo_espalda <= 190


# ── Función para verificar posición de caderas ────────────────────────────────
def verificar_caderas(cadera_y, hombro_y):
    """
    Verifica que las caderas no estén muy arriba o muy abajo
    """
    diferencia = abs(cadera_y - hombro_y)
    return diferencia < 150


# ── Función principal - VERSIÓN PARA COLAB/JUPYTER ────────────────────────────
def detector_flexiones_webcam():
    """
    Sistema completo de detección de flexiones con feedback en tiempo real
    VERSIÓN COMPATIBLE CON GOOGLE COLAB
    """
    
    # Importar librerías para Jupyter
    try:
        from IPython.display import display, Image as IPImage, clear_output
        from google.colab.patches import cv2_imshow
        JUPYTER = True
        print("✅ Modo Jupyter/Colab detectado")
    except:
        JUPYTER = False
        print("✅ Modo local detectado")
    
    # ── Configuración ─────────────────────────────────────────────────────────
    model = YOLO("yolo11n-pose.pt")
    
    # Webcam
    cap = cv2.VideoCapture(0)
    
    # Configurar resolución
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    # Verificar que la webcam se abrió
    if not cap.isOpened():
        print("❌ Error: No se pudo abrir la webcam")
        print("💡 Asegúrate de:")
        print("   1. Dar permisos a la cámara")
        print("   2. Verificar que no esté en uso por otra app")
        return
    
    # Variables de estado
    contador_correctas = 0
    contador_incorrectas = 0
    estado = None
    estado_ant = None
    
    print("🎥 Webcam iniciada.")
    print("📌 Colócate de perfil a la cámara para mejor detección.")
    
    if JUPYTER:
        print("⚠️  Para detener: Interrumpe el kernel (botón ⏹)")
    else:
        print("⏸  Para detener: Presiona 'q'")
    
    print()
    
    # ── Bucle principal ───────────────────────────────────────────────────────
    frame_count = 0
    
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                print("❌ Error al leer frame de la webcam")
                break
            
            frame_count += 1
            
            # Procesar cada 2 frames para mejor rendimiento
            if frame_count % 2 != 0:
                continue
            
            # Voltear la imagen horizontalmente (efecto espejo)
            frame = cv2.flip(frame, 1)
            
            # Detectar pose
            results = model(frame, verbose=False)
            
            # Reiniciar variables de feedback
            errores = []
            flexion_correcta = True
            
            for r in results:
                if r.keypoints is None or r.keypoints.xy.shape[0] == 0:
                    continue
                
                kpts = r.keypoints.xy[0]
                conf = r.keypoints.conf[0]
                
                # Índices de keypoints
                IDX_HOMBRO_DER = 6
                IDX_CODO_DER = 8
                IDX_MUNECA_DER = 10
                IDX_CADERA_DER = 12
                IDX_RODILLA_DER = 14
                IDX_TOBILLO_DER = 16
                
                # Verificar que todos los puntos sean visibles
                puntos_necesarios = [IDX_HOMBRO_DER, IDX_CODO_DER, IDX_MUNECA_DER, 
                                    IDX_CADERA_DER, IDX_RODILLA_DER, IDX_TOBILLO_DER]
                
                if all(conf[idx] > 0.5 for idx in puntos_necesarios):
                    
                    # Extraer coordenadas
                    hombro = (int(kpts[IDX_HOMBRO_DER][0]), int(kpts[IDX_HOMBRO_DER][1]))
                    codo = (int(kpts[IDX_CODO_DER][0]), int(kpts[IDX_CODO_DER][1]))
                    muneca = (int(kpts[IDX_MUNECA_DER][0]), int(kpts[IDX_MUNECA_DER][1]))
                    cadera = (int(kpts[IDX_CADERA_DER][0]), int(kpts[IDX_CADERA_DER][1]))
                    rodilla = (int(kpts[IDX_RODILLA_DER][0]), int(kpts[IDX_RODILLA_DER][1]))
                    tobillo = (int(kpts[IDX_TOBILLO_DER][0]), int(kpts[IDX_TOBILLO_DER][1]))
                    
                    # ── ANÁLISIS DE LA FLEXIÓN ────────────────────────────────────
                    
                    # 1. Ángulo del codo
                    angulo_codo = calcular_angulo(hombro, codo, muneca)
                    
                    # 2. Verificar alineación corporal
                    alineacion_correcta = verificar_alineacion(hombro, cadera, tobillo)
                    
                    # 3. Verificar posición de caderas
                    caderas_correctas = verificar_caderas(cadera[1], hombro[1])
                    
                    # 4. Verificar rodillas extendidas
                    angulo_rodilla = calcular_angulo(cadera, rodilla, tobillo)
                    rodillas_extendidas = angulo_rodilla > 160
                    
                    # ── DETERMINAR ESTADO ─────────────────────────────────────────
                    
                    if angulo_codo > 160:
                        estado = "arriba"
                    elif angulo_codo < 90:
                        estado = "abajo"
                    
                    # ── VALIDAR FORMA CORRECTA ────────────────────────────────────
                    
                    flexion_correcta = True
                    
                    if not alineacion_correcta:
                        errores.append("¡Espalda recta!")
                        flexion_correcta = False
                    
                    if not caderas_correctas:
                        errores.append("¡Caderas alineadas!")
                        flexion_correcta = False
                    
                    if not rodillas_extendidas:
                        errores.append("¡Extiende las piernas!")
                        flexion_correcta = False
                    
                    # ── CONTAR REPETICIONES ───────────────────────────────────────
                    
                    if estado_ant == "abajo" and estado == "arriba":
                        if flexion_correcta:
                            contador_correctas += 1
                            print(f"✅ Flexión CORRECTA #{contador_correctas}")
                        else:
                            contador_incorrectas += 1
                            print(f"❌ Flexión INCORRECTA (Errores: {', '.join(errores)})")
                    
                    estado_ant = estado
                    
                    # ── COLORES SEGÚN ESTADO ──────────────────────────────────────
                    
                    if flexion_correcta:
                        color_principal = (0, 255, 0)  # Verde
                        color_texto = (0, 255, 0)
                    else:
                        color_principal = (0, 0, 255)  # Rojo
                        color_texto = (0, 0, 255)
                    
                    if angulo_codo > 160:
                        color_codo = (0, 255, 0)
                    elif angulo_codo < 90:
                        color_codo = (255, 0, 0)
                    else:
                        color_codo = (0, 255, 255)
                    
                    # ── DIBUJAR SKELETON ──────────────────────────────────────────
                    
                    # Brazo
                    cv2.line(frame, hombro, codo, color_principal, 4)
                    cv2.line(frame, codo, muneca, color_principal, 4)
                    
                    # Torso
                    color_torso = (0, 255, 0) if alineacion_correcta else (0, 0, 255)
                    cv2.line(frame, hombro, cadera, color_torso, 4)
                    
                    # Piernas
                    color_piernas = (0, 255, 0) if rodillas_extendidas else (0, 0, 255)
                    cv2.line(frame, cadera, rodilla, color_piernas, 4)
                    cv2.line(frame, rodilla, tobillo, color_piernas, 4)
                    
                    # Círculos en articulaciones
                    cv2.circle(frame, hombro, 8, (255, 0, 0), -1)
                    cv2.circle(frame, codo, 8, color_codo, -1)
                    cv2.circle(frame, muneca, 8, (255, 0, 0), -1)
                    cv2.circle(frame, cadera, 8, (255, 255, 0), -1)
                    cv2.circle(frame, rodilla, 8, (255, 255, 0), -1)
                    cv2.circle(frame, tobillo, 8, (255, 255, 0), -1)
                    
                    # ── TEXTO INFORMATIVO ─────────────────────────────────────────
                    
                    cv2.putText(frame, f"{int(angulo_codo)}°",
                               (codo[0] - 40, codo[1] - 15),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.8, color_codo, 2)
                    
                    cv2.putText(frame, f"Estado: {estado if estado else 'Posicionate'}",
                               (20, 50),
                               cv2.FONT_HERSHEY_SIMPLEX, 1.0, color_texto, 2)
            
            # ── PANEL DE INFORMACIÓN ──────────────────────────────────────────────
            
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 80), (500, 300), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
            
            cv2.putText(frame, f"Correctas: {contador_correctas}",
                       (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
            
            cv2.putText(frame, f"Incorrectas: {contador_incorrectas}",
                       (20, 170), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
            
            total = contador_correctas + contador_incorrectas
            cv2.putText(frame, f"Total: {total}",
                       (20, 220), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
            
            if errores:
                y_pos = 270
                for error in errores:
                    cv2.putText(frame, f"⚠ {error}",
                               (20, y_pos),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    y_pos += 35
            else:
                cv2.putText(frame, "✓ Forma correcta!",
                           (20, 270),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            
            # ── MOSTRAR FRAME ─────────────────────────────────────────────────────
            
            if JUPYTER:
                # Versión para Jupyter/Colab
                _, buffer = cv2.imencode('.jpg', frame)
                clear_output(wait=True)
                display(IPImage(data=buffer.tobytes()))
            else:
                # Versión para entorno local
                cv2.imshow('Detector de Flexiones', frame)
                
                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'):
                    print("\n🛑 Deteniendo detector...")
                    break
                elif key == ord('r'):
                    contador_correctas = 0
                    contador_incorrectas = 0
                    print("\n🔄 Contadores reseteados")
    
    except KeyboardInterrupt:
        print("\n⏸  Detenido por el usuario")
    
    finally:
        # ── FINALIZAR ─────────────────────────────────────────────────────────
        cap.release()
        
        # ❌ NO USAR cv2.destroyAllWindows() en Colab
        if not JUPYTER:
            try:
                cv2.destroyAllWindows()
            except:
                pass
        
        print(f"\n📊 RESUMEN FINAL:")
        print(f"   ✅ Flexiones correctas: {contador_correctas}")
        print(f"   ❌ Flexiones incorrectas: {contador_incorrectas}")
        print(f"   📈 Total: {contador_correctas + contador_incorrectas}")
        
        if contador_correctas + contador_incorrectas > 0:
            porcentaje = (contador_correctas / (contador_correctas + contador_incorrectas)) * 100
            print(f"   🎯 Porcentaje de precisión: {porcentaje:.1f}%")


# ── EJECUTAR ──────────────────────────────────────────────────────────────────
detector_flexiones_webcam()

✅ Modo local detectado
❌ Error: No se pudo abrir la webcam
💡 Asegúrate de:
   1. Dar permisos a la cámara
   2. Verificar que no esté en uso por otra app


[ WARN:0@306.444] global cap_v4l.cpp:999 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ERROR:0@306.444] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range
